In [17]:
# standard library
import os
import re
from datetime import datetime
from pathlib import Path

# arrays / tables / plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import seaborn as sns
import xarray as xr

# geospatial
import geopandas as gpd
import folium
import contextily as ctx
import rasterio
from rasterio.plot import show
from rasterio.features import rasterize

# shapely geometry / operations
from shapely.geometry import (Point, LineString, MultiPoint, GeometryCollection, mapping,)
import shapely.ops as ops

# stats / interpolation / geostats
import scipy.interpolate
import scipy.stats
import skgstat as skg
import tqdm

# spatial stats
import pysal
from pysal.explore import esda
from pysal.lib import weights
from splot.esda import moran_scatterplot

# machine learning
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# cartographic helpers
from matplotlib_scalebar.scalebar import ScaleBar

In [18]:
#Summon the Data
DATA = Path("/home/jovyan/Society_of_Bouy_Cowboys/Data")

NETCDF = DATA / "NET_CDF"
ICE = DATA / "Ice_edge"
SAT = DATA / "SAT_GeoTiff"

# buoy files
moor_files = {
    "S1P1": NETCDF / "S1P1.nc",
    "S1P2": NETCDF / "S1P2.nc",
    "S1P3": NETCDF / "S1P3.nc",
    "S1A1": NETCDF / "S1A1.nc"
}



In [ ]:
#Create XArray for all GeoTiffs with Ice Edge

# Load all GeoTIFFs into a single xarray Dataset with a time dimension
scenes = {}
for tif_path in sorted(SAT.glob("*.tif")):
    m = re.search(r"_(\d{8})(\d{4})_", tif_path.stem)
    if m:
        t = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M")
    else:
        continue
    with rasterio.open(tif_path) as src:
        scenes[t] = {
            "sigma0":    src.read(1),
            "png":       src.read(2),
            "bounds":    src.bounds,
            "transform": src.transform,
            "shape":     (src.height, src.width),
        }

# Parse datetime from filename e.g. ice_edge_201911260350.geojson
ice_edges = {}
for geojson_path in sorted(ICE.glob("*.geojson")):
    m = re.search(r"_(\d{8})(\d{4})", geojson_path.stem)
    if m:
        t = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M")
        ice_edges[t] = gpd.read_file(geojson_path).to_crs("EPSG:32604")

print(f"Loaded {len(scenes)} SAR scenes")
print(f"Loaded {len(ice_edges)} ice edge files")

# ── Sort by time ───────────────────────────────────────────────────────────
times  = sorted(scenes.keys())
bounds = scenes[times[0]]["bounds"]
shape  = scenes[times[0]]["shape"]

# ── Rasterize ice edges onto the SAR grid ──────────────────────────────────
# For each SAR scene, if a matching ice edge exists (within 1 hour),
# burn it into a binary raster (1 = ice edge, 0 = no data).
# If no ice edge exists for that scene, fill with NaN.

def find_nearest_edge(t, ice_edges, max_hours=1):
    """Return the ice edge GeoDataFrame closest in time to t, within max_hours."""
    best_t, best_dt = None, None
    for et in ice_edges:
        dt = abs((t - et).total_seconds()) / 3600
        if dt <= max_hours and (best_dt is None or dt < best_dt):
            best_t, best_dt = et, dt
    return ice_edges[best_t] if best_t else None

ice_rasters = []
for t in times:
    edge_gdf = find_nearest_edge(t, ice_edges, max_hours=1)
    if edge_gdf is not None and len(edge_gdf) > 0:
        # Burn the line geometry into the raster grid
        burned = rasterize(
            [(mapping(geom), 1) for geom in edge_gdf.geometry],
            out_shape = shape,
            transform = scenes[t]["transform"],
            fill      = 0,
            dtype     = np.float32,
        )
    else:
        burned = np.full(shape, np.nan, dtype=np.float32)
    ice_rasters.append(burned)

print(f"Ice edge rasters built: {sum(np.any(r == 1) for r in ice_rasters)}/{len(times)} scenes have an ice edge")

# ── Build xarray Dataset ───────────────────────────────────────────────────
SAT_ds = xr.Dataset(
    data_vars=dict(
        SAR_backscatter=(["time", "y", "x"],
                np.stack([scenes[t]["sigma0"] for t in times]),
                {"long_name": "SAR backscatter", "units": "linear"}),
        ice_edge       =(["time", "y", "x"],
                np.stack(ice_rasters),
                {"long_name": "Ice edge (rasterized)", "units": "binary 0/1"}),
    ),
    coords=dict(
        time=(["time"], np.array(times, dtype="datetime64[ns]")),
        x   =(["x"],    np.linspace(bounds.left,   bounds.right, shape[1])),
        y   =(["y"],    np.linspace(bounds.top,     bounds.bottom, shape[0])),
    ),
    attrs=dict(crs="EPSG:32604"),
)

SAT_ds

Loaded 25 SAR scenes
Loaded 8 ice edge files
Ice edge rasters built: 9/25 scenes have an ice edge


In [ ]:
#Organize Data Moorings + Put into Coordinate System
def get_mooring_latlon(ncfile):
    ds = xr.open_dataset(ncfile)

    lat = np.asarray(ds["lat_lagrangian"]).astype(float).ravel()
    lon = np.asarray(ds["lon_lagrangian"]).astype(float).ravel()

    good = np.isfinite(lat) & np.isfinite(lon)

    lat = lat[good]
    lon = lon[good]

    if len(lat) == 0:
        raise ValueError(f"No valid lat/lon found in {ncfile}")

    # median gives robust representative location
    return np.median(lat), np.median(lon)

rows = []

for name, fn in moor_files.items():
    lat, lon = get_mooring_latlon(fn)
    rows.append({
        "source": name,
        "lat": lat,
        "lon": lon
    })

moor_df = pd.DataFrame(rows)

moor_gdf = gpd.GeoDataFrame(
    moor_df,
    geometry=gpd.points_from_xy(moor_df["lon"], moor_df["lat"]),
    crs="EPSG:4326"
)

# convert buoys into analysis CRS
moor_gdf = moor_gdf.to_crs("EPSG:32604")

moor_gdf

In [ ]:
#Plot to Check Work
Nov_26 = SAT_ds.sel(time="2019-11-26", method="nearest")

fig, ax = plt.subplots(figsize=(8, 8))

# SAR image
Nov_26["SAR_backscatter"].plot(ax=ax, cmap="gray", add_colorbar=False)

# Ice edge — plot only where value == 1
ice = Nov_26["ice_edge"].values
if not np.all(np.isnan(ice)):
    ax.contour(
        SAT_ds.x, SAT_ds.y, ice,
        levels=[0.5],
        colors="red",
        linewidths=2,
        zorder=4,
        label="Ice edge"
    )
    ax.plot([], [], color="red", linewidth=2, label="Ice edge")  # legend entry

# Moorings
moor_gdf.plot(ax=ax, color="yellow", markersize=50, zorder=5, label="Moorings")

# Labels + title
ax.set_title("Sea Ice Edge and Mooring Locations\n" + str(Nov_26.time.values)[:16],fontsize=14)
ax.set_xlabel("(m)")
ax.set_ylabel("(m)")
ax.legend();

In [ ]:
moor_gdf.to_file("/home/jovyan/Society_of_Bouy_Cowboys/Data/moorings.geojson", driver="GeoJSON")

In [ ]:
# ===========================================================================
# CONFIG
# ===========================================================================

OUTPUT_PATH = "/home/jovyan/Society_of_Bouy_Cowboys/timelapse.gif"  # or .mp4
FPS         = 2
FIGSIZE     = (8, 8)
DPI         = 120

# ===========================================================================


# ── Compute the tight bounding box across all scenes ──────────────────────
# Only include pixels that actually have data (non-zero, non-NaN) to cut
# out blank space around the edges.

sar = SAT_ds["SAR_backscatter"].values   # (time, y, x)
xs  = SAT_ds.x.values
ys  = SAT_ds.y.values

# Find columns and rows that have at least one valid pixel across all frames
valid_mask  = np.any(np.isfinite(sar) & (sar > 0), axis=0)
valid_rows  = np.where(valid_mask.any(axis=1))[0]
valid_cols  = np.where(valid_mask.any(axis=0))[0]

row_min, row_max = valid_rows[0],  valid_rows[-1]
col_min, col_max = valid_cols[0],  valid_cols[-1]

x_min = xs[col_min];  x_max = xs[col_max]
y_min = ys[row_max];  y_max = ys[row_min]   # ys go north-down

# ── Build persistent canvas ────────────────────────────────────────────────
# Canvas is updated in-place: new data overwrites old, blanks keep old value.
canvas = np.full((row_max - row_min + 1,
                  col_max - col_min + 1), np.nan, dtype=np.float32)

# ── Build per-frame data ───────────────────────────────────────────────────
n_frames   = len(SAT_ds.time)
times      = SAT_ds.time.values
ice_layers = SAT_ds["ice_edge"].values   # (time, y, x)

frames_canvas    = []
frames_ice       = []

last_ice = None   # most recent ice edge, persists until a new one arrives

for i in range(n_frames):
    # Update canvas with new SAR data
    new_sar  = sar[i, row_min:row_max+1, col_min:col_max+1]
    has_data = np.isfinite(new_sar) & (new_sar > 0)
    canvas   = canvas.copy()
    canvas[has_data] = new_sar[has_data]
    frames_canvas.append(canvas.copy())

    # Update ice edge — keep last known edge if current frame has none
    ice_slice = ice_layers[i, row_min:row_max+1, col_min:col_max+1]
    if np.any(ice_slice == 1):
        last_ice = ice_slice.copy()
    frames_ice.append(last_ice)   # None if no ice edge seen yet

# ── Convert mooring points to axis coordinates ────────────────────────────
# moor_32604 must be defined and in EPSG:32604 (metres)
moor_xs = [geom.x for geom in moor_gdf.geometry]
moor_ys = [geom.y for geom in moor_gdf.geometry]
moor_labels = list(moor_gdf["source"])

# ── Build animation ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

# Compute global vmin/vmax for consistent colour scale across all frames
all_valid = np.concatenate([f[np.isfinite(f) & (f > 0)].ravel()
                             for f in frames_canvas])
vmin = float(np.percentile(all_valid, 2))
vmax = float(np.percentile(all_valid, 98))

# xs/ys cropped to tight bbox
xs_crop = xs[col_min:col_max+1]
ys_crop = ys[row_min:row_max+1]

def draw_frame(i):
    ax.clear()

    # SAR canvas
    ax.imshow(
        frames_canvas[i],
        cmap   = "gray",
        vmin   = vmin,
        vmax   = vmax,
        origin = "upper",
        extent = [x_min, x_max, y_min, y_max],
        aspect = "auto",
    )

    # Ice edge
    if frames_ice[i] is not None:
        ax.contour(
            xs_crop, ys_crop, frames_ice[i],
            levels     = [0.5],
            colors     = "red",
            linewidths = 2,
            zorder     = 4,
        )
        ax.plot([], [], color="red", linewidth=2, label="Ice edge")

    # Mooring points
    ax.scatter(moor_xs, moor_ys,
               color="yellow", s=60, zorder=5, label="Moorings")
    for x, y, lbl in zip(moor_xs, moor_ys, moor_labels):
        ax.annotate(lbl, xy=(x, y), xytext=(5, 5),
                    textcoords="offset points",
                    color="yellow", fontsize=7)

    # Labels
    t_str = str(times[i])[:16].replace("T", "  ")
    ax.set_title(t_str, fontsize=10)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.text(0.98, 0.02, f"{i+1}/{n_frames}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=7, color="white",
            bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5))
    ax.legend(loc="lower right", fontsize=8)

ani = animation.FuncAnimation(
    fig, draw_frame, frames=n_frames, interval=int(1000 / FPS)
)

# ── Save ───────────────────────────────────────────────────────────────────
ext = OUTPUT_PATH.split(".")[-1].lower()
if ext == "gif":
    ani.save(OUTPUT_PATH, writer="pillow", fps=FPS)
elif ext == "mp4":
    ani.save(OUTPUT_PATH, writer="ffmpeg", fps=FPS)

print(f"Saved to {OUTPUT_PATH}")

plt.close()
HTML(ani.to_jshtml())